# Лекция: Нейронные сети и глубокое обучение

Далее потребуются следующие модули Python. Убедитесь, что они установлены.
- `matplotlib`
- `requests`
- `numpy`
- `tensorflow`

## Нейроны

Искусственная нейронная сеть — это модель машинного обучения, идея которой основана на принципах работы мозга.

Как и мозг, искусственная нейронная сеть представляет собой совокупность нейронов.


![640px-Neuron3.svg.png](https://raw.githubusercontent.com/kupav/data-sc-intro/refs/heads/main/fig/640px-Neuron3.svg.png)

![art_neuron.png](https://raw.githubusercontent.com/kupav/data-sc-intro/refs/heads/main/fig/art_neuron.png)

Каждый нейрон собирает сигналы от других нейронов, суммирует их, каждый сигнал со своим весом, а затем выдает выход через нелинейную функцию $\sigma$.

$$
y=\sigma(xw+b)
$$

Здесь $w$ — вектор весов входных сигналов, а $b$ — смещение (bias).

Чтобы найти выход, сначала вычисляется скалярное произведение вектора входных сигналов $x$ и его весов $w$:

$$
x w = (x_1, x_2, x_3, \dots) (w_1, w_2, w_3, \dots) = x_1 w_1 + x_2 w_2 + x_3 w_3 + \ldots
$$

Результат называется взвешенной суммой входов.

Затем к ней прибавляется скалярное смещение $b$. Оно необходимо для установки порога активации нейрона.

Функция $\sigma$, которая применяется после добавления смещения, называется функцией активации.

В простейшем случае активация — это ступенчатая функция:

In [ ]:
def stepfunc(x):
    return 1.0 if x >= 0 else 0.0

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

xs = np.linspace(-3, 3, 100)
ys = [stepfunc(x) for x in xs]

fig, ax = plt.subplots()
ax.plot(xs, ys)
ax.set_xlabel(r"x")
ax.set_ylabel(r"y")
ax.grid()

Эта функция называется «активацией», потому что предполагается, что когда входное значение ниже порога, нейрон остается спокойным ($y=0$), а если вход превысил порог, нейрон активируется ($y=1$).

Порог задается значением $b$. Если взвешенная сумма входов $xw$ меньше $-b$, то прибавление $b$ дает отрицательный результат, и нейрон остается спокойным:

$$
\text{если }(xw < -b) \text{ то } (xw + b < 0)
$$

А если взвешенная сумма больше $-b$, нейрон активируется:

$$
\text{если }(xw > -b) \text{ то } (xw + b > 0)
$$

Изменяя $b$, мы управляем порогом активации нейрона.

Хотя часто используются и другие функции активации, у большинства из них есть две части: одна соответствует спокойному нейрону, другая — активированному.

Обычно нейронная сеть содержит множество связанных нейронов.

Они организованы в слои: входной слой принимает входные сигналы, его выход является входным сигналом для следующего слоя и так далее. Последний слой выдает общий выход всей сети.

Нейронные сети могут решать самые разные задачи, такие как распознавание рукописного текста и обнаружение лиц, анализ и генерация текста.

Чтобы сеть могла решить задачу, ее необходимо обучить. Обучение включает настройку значений весов $w$ и смещений $b$.

В результате получается «черный ящик». Изучение обученных значений $w$ и $b$ не дает особого понимания того, как они решают задачу.


## Один нейрон


In [ ]:
import csv
import numpy as np
import requests

def load_csv_dataset(file_name, dtype=float):
    """Downloads csv numeric dataset from repo to numpy array."""
    base_url = "https://raw.githubusercontent.com/kupav/data-sc-intro/main/data/"
    web_data = requests.get(base_url + file_name)
    assert web_data.status_code == 200

    reader = csv.reader(web_data.text.splitlines(), delimiter=',')
    data = []
    for row in reader:
        try:
            # Try to parse as a row of floats
            float_row = [dtype(x) for x in row]
            data.append(float_row)
        except ValueError:
            # If parsing as floats failed - this is header
            print(row)

    return np.array(data)

Рассмотрим самый простой пример нейронной сети, включающей только один нейрон.

Давайте создадим его и посмотрим, как он работает.

Для этого мы будем использовать набор данных `perceptron_data.csv`

In [ ]:
data = load_csv_dataset('perceptron_data.csv')
print(data.shape)

Мы видим, что набор данных содержит 400 строк и два столбца.

Визуализируем его.


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5,5))
ax.scatter(data[:, 0], data[:, 1]);
ax.axhline(y=5, color='k')
ax.axvline(x=0, color='k')
ax.set_xlabel(r"$x_1$")
ax.set_ylabel(r"$x_2$")
ax.grid()

Точки четко сгруппированы в четыре кластера.

В качестве первого шага мы создадим сеть, способную идентифицировать пару групп, разделенных линией $x_1=0$:

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5,5))
ax.scatter(data[:, 0], data[:, 1]);
ax.axvline(x=0, color='k')
ax.set_xlabel(r"$x_1$")
ax.set_ylabel(r"$x_2$")
ax.grid()

Наша сеть из одного нейрона должна определять, находится ли точка в правой полуплоскости или в левой.

Создадим ее:


In [ ]:
def theneuron(x, prm):
    """
    prm is a dictionary with two keys
    prm['w'] - vector of weights
    prm['b'] - scalar bias
    """
    x = np.array(x, dtype=float)
    w = np.array(prm['w'], dtype=float)
    b = prm['b']
    y = x @ w + b
    return 1 if y >= 0 else 0

Чтобы решить нашу простую задачу, мы легко можем найти веса $w$ и смещение $b$ вручную, без необходимости применять обучение.

Можно легко догадаться, что $w=(1, 0)$ и $b=0$ подходят.

Проверим это. Предскажем метки, используя предложенные $w$ и $b$:

In [ ]:
prm0 = {'w': [1.0, 0.0], 'b': 0}

y_pred = [theneuron(x, prm0) for x in data]
print(y_pred)

In [ ]:
import matplotlib.pyplot as plt

colors = ['r', 'b']

fig, ax = plt.subplots(figsize=(5,5))
ax.scatter(data[:, 0], data[:, 1], color=[colors[y] for y in y_pred]);
ax.axvline(x=0, color='k')
ax.set_xlabel(r"$x_1$")
ax.set_ylabel(r"$x_2$")
ax.grid()

Мы видим, что наш нейрон работает отлично.

Более сложная задача: найдем параметры нейрона, который способен идентифицировать верхнюю правую группу точек.

Другими словами, давайте разделим наш набор данных линией, подобной этой:


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5,5))
ax.scatter(data[:, 0], data[:, 1]);
ax.axline((-0.5, 6.5), (1.5, 4.5), color='k')
ax.set_xlabel(r"$x_1$")
ax.set_ylabel(r"$x_2$")
ax.grid()

Нам нужно найти такие $w_1$, $w_2$ и $b$, чтобы $xw+b$ обращалось в ноль вдоль этой линии.

В этом случае все точки ниже нее будут классифицированы нейроном как принадлежащие к классу 0, а те, что выше, — к классу 1.

У нас есть три неизвестных переменных: $w_1$, $w_2$ и $b$. Это означает, что нам нужно три точки вдоль разделяющей линии, чтобы записать три уравнения для неизвестных.

Изучая график выше, мы легко можем найти эти точки:

$$
(x_1, x_2) = (-0.5, 6.5), (1.5, 4.5), (0.5, 5.5)
$$

Подставим их в уравнение нейрона $x_1 w_1 + x_2 w_2 + b = 0$:

$$
  \left\{
    \begin{aligned}
      {}& -0.5 w_1 + 6.5 w_2 + b = 0 \\
      {}& \phantom{+}1.5 w_1 + 4.5 w_2 + b = 0 \\
      {}& \phantom{+}0.5 w_1 + 5.5 w_2 + b = 0
    \end{aligned}
  \right.
$$

Эту систему уравнений можно переписать в матричной форме:

$$
\begin{pmatrix}
  -0.5 & 6.5 & 1 \\
  1.5  & 4.5 & 1 \\
  0.5  & 5.5 & 1
\end{pmatrix}
\begin{pmatrix}w_1 \\ w_2 \\ b\end{pmatrix} =
\begin{pmatrix}0 \\ 0 \\ 0\end{pmatrix}
$$

Это уравнение можно рассматривать как задачу на собственные значения матрицы:

Дана матрица $M$, нужно найти такой вектор $v$, чтобы его умножение на матрицу $M$ приводило к его растяжению (или сжатию) на $\lambda$:

$$
M v = \lambda v
$$

Здесь $\lambda$ — скаляр, который необходимо найти. $\lambda$ называется собственным значением, а $v$ — собственным вектором матрицы.

В нашем случае

$$
M=
\begin{pmatrix}
  -0.5 & 6.5 & 1 \\
  1.5  & 4.5 & 1 \\
  0.5  & 5.5 & 1
\end{pmatrix}
$$

и нам нужно особое собственное значение $\lambda=0$.

Конечно, не гарантируется, что у этой матрицы есть нулевое собственное значение. В нашем случае это означало бы, что мы взяли неправильные точки.

Вычисление собственных значений и векторов очень просто с помощью `numpy`. Мы просто используем функцию `np.linalg.eig`:


In [ ]:
M = np.array([[-0.5, 6.5, 1],[1.5, 4.5, 1],[0.5, 5.5, 1]])
lam, V = np.linalg.eig(M)
print("Eigenvalues of M are: ", lam)
print("Each column of V is an eigenvector of M:\n", V)

Мы видим, что одно из собственных значений очень близко к нулю. Это то, что нам нужно: точный ноль не может быть получен из-за численных погрешностей.

Соответствующий собственный вектор дает требуемые параметры нейрона.

Обратите внимание, что мы можем изменить знаки всех элементов собственного вектора, и он все равно останется собственным вектором.

Мы хотим обозначить верхнюю правую группу как класс 1, а остальные — как класс 0.

Это произойдет, если мы изменим знаки коэффициентов.

In [ ]:
w1, w2, b = V[:, 2]
prm1 = {'w': [-w1, -w2], 'b': -b}
print(prm1)

Проверим, как наш нейрон классифицирует точки набора данных.


In [ ]:
y_pred = [theneuron(x, prm1) for x in data]
print(y_pred)

In [ ]:
import matplotlib.pyplot as plt

colors = ['r', 'b']

fig, ax = plt.subplots(figsize=(5,5))
ax.scatter(data[:, 0], data[:, 1], color=[colors[y] for y in y_pred]);
ax.axline((-0.5, 6.5), (1.5, 4.5), color='k')
ax.set_xlabel(r"$x_1$")
ax.set_ylabel(r"$x_2$")
ax.grid()

Теперь попробуем найти нейрон, который выделяет нижнюю левую группу точек.


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5,5))
ax.scatter(data[:, 0], data[:, 1]);
ax.axline((-1.5, 5.5), (0.5, 3.5), color='k')
ax.set_xlabel(r"$x_1$")
ax.set_ylabel(r"$x_2$")
ax.grid()

Три точки, которые нам нужны для этого:

$$
(x_1, x_2) = (-1.5, 5.5), (0.5, 3.5), (-0.5, 4.5)
$$

Уравнения для неизвестных коэффициентов имеют вид:

$$
\begin{pmatrix}
  -1.5 & 5.5 & 1 \\
   0.5 & 3.5 & 1 \\
  -0.5 & 4.5 & 1
\end{pmatrix}
\begin{pmatrix}w_1 \\ w_2 \\ b\end{pmatrix} =
\begin{pmatrix}0 \\ 0 \\ 0\end{pmatrix}
$$

Мы готовы вычислить собственные значения матрицы. Коэффициенты соответствуют нулевому собственному значению.


In [ ]:
M = np.array([[-1.5, 5.5, 1.0], [0.5, 3.5, 1.0], [-0.5, 4.5, 1.0]])
lam, V = np.linalg.eig(M)
print("Eigenvalues of M are: ", lam)
print("Each column of V is an eigenvector of M:\n", V)

In [ ]:
w1, w2, b = V[:, 1]
prm2 = {'w': [w1, w2], 'b': b}
print(prm2)

Проверим, как нейрон классифицирует точки набора данных.


In [ ]:
y_pred = [theneuron(x, prm2) for x in data]
print(y_pred)

In [ ]:
import matplotlib.pyplot as plt

colors = ['r', 'b']

fig, ax = plt.subplots(figsize=(5,5))
ax.scatter(data[:, 0], data[:, 1], color=[colors[y] for y in y_pred]);
ax.axline((-1.5, 5.5), (0.5, 3.5), color='k')
ax.set_xlabel(r"$x_1$")
ax.set_ylabel(r"$x_2$")
ax.grid()

Одиночный нейрон способен классифицировать данные тогда и только тогда, когда они могут быть разделены одной прямой линией. Или если у нас многомерный набор данных — гиперплоскостью.

Одиночный нейрон не способен выполнить свою работу, если у нас есть две группы, подобные этим:

In [ ]:
import matplotlib.pyplot as plt

colors = ['r', 'b']

y_manual = [1 if x[0]*(x[1]-5) > 0 else 0 for x in data]

fig, ax = plt.subplots(figsize=(5,5))
ax.scatter(data[:, 0], data[:, 1], color=[colors[y] for y in y_manual]);
ax.axline((-0.5, 6.5), (1.5, 4.5), color='k')
ax.axline((-1.5, 5.5), (0.5, 3.5), color='k')
ax.set_xlabel(r"$x_1$")
ax.set_ylabel(r"$x_2$")
ax.grid()

Это ограничение одиночного нейрона известно как «проблема XOR», потому что его можно рассматривать с точки зрения моделирования логических функций.

Одиночный нейрон может моделировать логические И, ИЛИ и НЕ, но не может моделировать XOR, т.е. исключающее ИЛИ.

Однако эту и многие другие, гораздо более сложные задачи можно решить с помощью нейронных сетей, включающих более одного нейрона и более одного слоя.

В дальнейшем мы будем рассматривать сети из многих нейронов.

Будет удобнее определить каждый нейрон в более компактной форме.

Сначала мы создадим класс для одиночного нейрона, а затем определим класс для слоя нейронов.


In [ ]:
class Neuron:
    """
    Neuron with step activation
    """
    def __init__(self, w, b):
        self.w = np.array(w)
        self.b = b

    def __str__(self):
        """String reprsentation to print weights"""
        return f"w={self.w}, b={self.b}"

    def stepfunc(y):
        return 1 if y >= 0 else 0

    def __call__(self, x):
        y = np.array(x) @ self.w + self.b
        return Neuron.stepfunc(y)

Пересоздадим наш нейрон, используя новый класс.


In [ ]:
pcp1 = Neuron(w=[0.16222142113076246, 0.16222142113076252], b=-0.9733285267845752)
print(pcp1)

Применим новую версию нейрона к набору данных следующим образом:

In [ ]:
y_pred = [pcp1(x) for x in data]
print(y_pred[:10])

In [ ]:
import matplotlib.pyplot as plt

colors = ['r', 'b']

fig, ax = plt.subplots(figsize=(5,5))
ax.scatter(data[:, 0], data[:, 1], color=[colors[y] for y in y_pred]);
ax.axline((-0.5, 6.5), (1.5, 4.5), color='k')
ax.set_xlabel(r"$x_1$")
ax.set_ylabel(r"$x_2$")
ax.grid()

## Слой нейронов

Одиночный нейрон не так интересен. Его возможности очень ограничены.

Гораздо интереснее, когда несколько нейронов соединены в сеть.

Типичная единица нейронной сети — это слой, набор нейронов, которые получают одни и те же входные сигналы и выдают независимые выходы.

Рассмотрим два нейрона, которые питаются одним и тем же входным вектором $x=(x_1, x_2, x_3)$. Предположим на мгновение, что активация отсутствует.

$$
y_1=(x_1, x_2, x_3) (w_{11}, w_{21}, w_{31}) + b_1 = x_1 w_{11} + x_2 w_{21} + x_3 w_{31} + b_1
$$

$$
y_2=(x_1, x_2, x_3) (w_{12}, w_{22}, w_{32}) + b_2 = x_1 w_{12} + x_2 w_{22} + x_3 w_{32} + b_2
$$

Здесь нам нужно вспомнить умножение вектора на матрицу. Эти два уравнения можно переписать более компактно следующим образом:

$$
(y_1, y_2) =
(x_1, x_2, x_3)
\begin{pmatrix}
w_{11} & w_{12} \\
w_{21} & w_{22} \\
w_{31} & w_{32}
\end{pmatrix}
+ (b_1, b_2)
$$

Обозначим матрицу как $W$:

$$
W =
\begin{pmatrix}
w_{11} & w_{12} \\
w_{21} & w_{22} \\
w_{31} & w_{32}
\end{pmatrix}
$$

И теперь $b$ рассматривается как вектор:

$$
b = (b_1, b_2)
$$

Выходы нейронов также можно рассматривать как вектор:

$$
y = (y_1, y_2)
$$

Теперь операцию этих двух нейронов можно представить следующим образом:

$$
y = x W + b
$$

Напомним наконец о функции активации. Когда мы применяем ее к векторам, предполагается поэлементная операция:

$$
y = \sigma(x W + b)
$$

Мы вывели эту формулу, предполагая, что $x$ — трехмерный вектор, а нейронов два. Но эту компактную форму можно записать для произвольного количества входов и любого количества нейронов.

Таким образом, приведенная выше формула представляет слой нейронов.

Ниже приведена иллюстрация слоя нейронов. На этом рисунке мы видим, что все входы соединены со всеми нейронами. Это называется плотным слоем (dense layer).

Полносвязные соединения не обязательны. Существуют сети, где каждый нейрон соединен с определенным подмножеством входов.


![art_neuron_layer.png](https://raw.githubusercontent.com/kupav/data-sc-intro/refs/heads/main/fig/art_neuron_layer.png)

Определим класс для слоя нейронов. Он очень похож на приведенный выше класс нейрона.

Обратите внимание, что мы передаем функцию активации как параметр конструктора.

In [ ]:
class Layer:
    def __init__(self, W, b, activat):
        # Number of rows of W equals to the number of input signals
        # Number of columns of W and number of elements of b equals to neurons in the layer
        self.W = np.array(W)
        self.b = np.array(b)
        # W must be a rectangular matrix (W.shape can be like (3, 5))
        assert len(W.shape) == 2
        # b must be a vector (b.shape can be like (5,))
        assert len(b.shape) == 1
        # Number of columns of W must coincide with number of elements of b
        assert W.shape[1] == b.shape[0]
        self.activat = activat

    def __str__(self):
        """String reprsentation to print weights"""
        return f"W=\n{self.W}\nb=\n{self.b}"

    def __call__(self, x):
        y = np.array(x) @ self.W + self.b
        return self.activat(y)

Нам нужно подготовить ступенчатую функцию, которая может работать с массивами.

Массив NumPy можно сравнить с константой следующим образом:

In [ ]:
tst = np.array([-1.2, 2.3])
tst >= 0

Результатом является массив из True и False.

В Python (и во всех других языках программирования) логические значения False и True на самом деле представлены целыми числами 0 и 1 соответственно.

Таким образом, мы получим желаемую ступенчатую функцию, если преобразуем результат сравнения в целые числа:


In [ ]:
(tst >= 0).astype(int)

Таким образом, подходящая ступенчатая функция для массивов numpy имеет вид:



In [ ]:
def stepfunc(y):
    return (y >= 0).astype(int)

## Двухслойная сеть прямого распространения

Создадим сеть, которая решает проблему XOR.

Сначала создадим слой из двух нейронов.

Один нейрон будет помечать точки в верхнем правом углу как 1, а остальные — как 0.

Другой будет помечать нижний левый угол как 1, а остальные — как 0.

Мы просто копируем коэффициенты из предыдущих примеров.

Обратите внимание, что мы записываем веса нейронов как строки для удобства, а затем транспонируем их, чтобы сформировать соответствующую матрицу $W$.


In [ ]:
W1 = np.array([[0.16222142113076246, 0.16222142113076252], [-0.23570226039551573, -0.23570226039551598]]).T
b1 = np.array([-0.9733285267845752, 0.9428090415820632])
lay1 = Layer(W1, b1, stepfunc)
print(lay1)

Проверим, как это работает. Сначала применим слой к набору данных, а затем преобразуем сигмоидные выходы в 0 или 1.


In [ ]:
y_pred = lay1(data)
print(y_pred[:10])

Чтобы убедиться, что все в порядке, построим две точечные диаграммы. Первая раскрашена в соответствии с нейроном 1, а вторая — в соответствии с нейроном 2.

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(11,5))

colors = ['r', 'b']

axs[0].scatter(data[:, 0], data[:, 1], color=[colors[y[0]] for y in y_pred])
axs[0].set_title("Labeled according to the neuron 1")
axs[1].scatter(data[:, 0], data[:, 1], color=[colors[y[1]] for y in y_pred])
axs[1].set_title("Labeled according to the neuron 2")

for ax in axs:
    ax.set_xlabel(r"$x_1$")
    ax.set_ylabel(r"$x_2$")
    ax.grid()

Посмотрим, какие комбинации сообщает наш слой.

Чтобы получить список уникальных значений, мы обычно преобразуем список данных в набор. Но теперь у нас есть список массивов numpy. Для него нельзя создать набор.

Сначала преобразуем наши данные в список кортежей.

In [ ]:
y_pred_tuple = [tuple(y) for y in y_pred]
print(y_pred_tuple[:10])

Теперь можно увидеть уникальные комбинации.


In [ ]:
print(set(y_pred_tuple))

Комбинации (1, 0) и (0, 1) указывают на точки в верхнем правом и нижнем левом углах соответственно. А (0, 0) соответствует двум другим группам.

Давайте вспомним, что мы хотим создать сеть, которая различает точки в верхнем правом и нижнем левом углах как один класс, а две другие — как другой класс.

Для этого мы добавляем еще один слой с одним нейроном, который получает выходы предыдущего слоя и суммирует их.

Сумма равна 1 для (1, 0) и (0, 1) и 0 для (0, 0).

Это оно. Нам даже не нужны $b$ и активация. Таким образом, мы устанавливаем $b=0$ и передаем в конструктор слоя пустую функцию активации.


In [ ]:
def identity(y):
    return y

In [ ]:
W2 = np.array([[1.0, 1.0]]).T
b2 = np.array([0])
lay2 = Layer(W2, b2, identity)
print(lay2)

Чтобы объединить два слоя, мы просто отправляем выход первого слоя как вход для второго:


In [ ]:
y_pred = lay2(lay1(data))
print(y_pred[:10])
print(y_pred.shape)

Теперь мы готовы показать, что наша двухслойная сеть способна различать группы точек, как требуется.

Обратите внимание, что выход сети представляет собой двумерный массив numpy из чисел с плавающей точкой.

И мы собираемся использовать его в качестве индексов цвета.

Для этого нам нужно взять первый (и единственный) элемент каждой строки и преобразовать его в целое число.


In [ ]:
colors = ['r', 'b']

y = y_pred[0]
print(y)

colors[int(y[0])]

In [ ]:
import matplotlib.pyplot as plt

colors = ['r', 'b']

fig, ax = plt.subplots(figsize=(5,5))
ax.scatter(data[:, 0], data[:, 1], color=[colors[int(y[0])] for y in y_pred]);
ax.axline((-0.5, 6.5), (1.5, 4.5), color='k')
ax.axline((-1.5, 5.5), (0.5, 3.5), color='k')
ax.set_xlabel(r"$x_1$")
ax.set_ylabel(r"$x_2$")
ax.grid()

Созданная нами сеть имеет следующую структуру.


![art_neuron_hidden.png](https://raw.githubusercontent.com/kupav/data-sc-intro/refs/heads/main/fig/art_neuron_hidden.png)

Она имеет входы и два слоя. Слой посередине не виден пользователям сети, так как его результат поступает на другой слой.

Этот слой называется скрытым. Последний слой, который выдает предсказание сети, называется выходным слоем.

В принципе, это минимальная структура сети, которая может практически все, что делают нейронные сети.

Существует математическая теорема, теорема об универсальной аппроксимации, которая утверждает, что этой структуры достаточно для всех целей.

Конечно, для разных задач мы должны варьировать количество входов и размеры слоев.

Однако на практике это неэффективно. Во многих случаях требуется очень большой скрытый слой, и настройка его параметров становится почти неразрешимой задачей.

Вот почему вместо очень широких сетей с огромным скрытым слоем сегодня используются глубокие сети со многими (на самом деле очень многими) слоями, идущими один за другим.

Сеть, в которой нейроны собраны в слои и каждый слой соединен со следующим, называется сетью прямого распространения (feed-forward network).

## Сигмоидальная функция активации

Ступенчатая функция в качестве активации неудобна.

Обучение нейронов требует вычисления производных, а ступенчатая функция для этого очень плохо подходит.

Ее производная равна нулю везде, кроме точки $x=0$, где производная стремится к бесконечности.

Обычно вместо ступенчатой функции используется ее гладкая аппроксимация.

Это называется сигмоидой.

$$
\sigma(x)=\frac{1}{1+e^{-x}}
$$


In [ ]:
def stepfunc(x):
    return 1.0 if x >= 0 else 0.0

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

xs = np.linspace(-10, 10, 100)
y_step = [stepfunc(x) for x in xs]
y_sigm = [sigmoid(x) for x in xs]

fig, ax = plt.subplots()
ax.plot(xs, y_step, label='step function')
ax.plot(xs, y_sigm, label='sigmoid')
ax.set_xlabel(r"x")
ax.set_ylabel(r"y")
ax.legend()
ax.grid()

Часто, когда обученная сеть работает в режиме предсказания, выход последнего слоя не передается в его функцию активации.

Выходы сети без последней активации называются логитами (logits):

$$
  y_\text{logits} = x W + b
$$

Когда сеть обучается, мы вычисляем активацию следующим образом:

$$
  y = \sigma(y_\text{logits})
$$

Но при предсказании мы можем вместо этого преобразовать логиты в точные 0 и 1.

Глядя на график сигмоиды, мы видим, что естественным порогом является $y_\text{logits}=0$. Если $y_\text{logits}<0$, мы можем заменить выход сигмоиды на $y=0$, а для $y_\text{logits}>0$ — на $y=1$.

Фактически, это преобразование логитов в 0 и 1 означает, что в режиме предсказания мы неявно используем ступенчатую функцию для выхода последнего слоя.

## Обучение сети с помощью обратного распространения ошибки

Ранее мы угадывали значения весов и смещений нейронов.

Но это было потому, что рассматриваемый случай был очень простым.

Для более серьезных случаев нам нужна процедура автоматической настройки параметров сети.

Это называется обучением сети, и обычно это делается с помощью метода градиентного спуска.

Давайте вспомним, что это такое.

Предположим, что $v_n=(x_n, y_n)$ — вектор, а $L(v_n)$ — функция, минимум которой нужно найти.

Алгоритм спуска к минимуму выглядит следующим образом:

$$
v_{n+1} = v_n - \gamma \nabla L(v_n)
$$

Это уравнение градиентного спуска. Итерации останавливаются, когда расстояние между двумя последовательными векторами становится достаточно маленьким или значение $L$ становится достаточно маленьким (в идеале — нулевым).

Математическая конструкция $\nabla L(v_n)$ называется градиентом $L(v_n)$.

Обучение на основе градиентного спуска называется обратным распространением (backpropagation).

Сначала нейронная сеть распространяет сигнал входных данных вперед через свои параметры к моменту принятия решения.

А затем она распространяет информацию об ошибке обратно через сеть, чтобы можно было изменить параметры. Это происходит шаг за шагом:

- Сеть получает входные данные и делает пробное предсказание
- Вычисляется функция потерь, которая сравнивает предсказание сети с желаемым
- Вычисляется ошибка и распространяется обратно для корректировки параметров

Корректировка параметров осуществляется с помощью метода градиентного спуска. Градиент функции потерь вычисляется по параметрам сети, и они корректируются, как показано в уравнении выше.

Для простоты мы обсудим обучение одного нейрона, чтобы он мог различать верхнюю правую группу точек нашего набора данных.

Пусть $x_{i1}$ и $x_{i2}$ — координаты $i$-й точки в наборе данных, а $y_i$ — желаемый класс этой точки: 0 или 1.

Рассмотрим функцию потерь, определенную как среднеквадратичная ошибка:

$$
L = \frac{1}{N} \sum_{i=1}^N \left[\sigma(x_{i1} w_1 + x_{i2} w_2 + b) - y_i\right]^2
$$

Мы хотим настроить параметры нейрона $w_1$, $w_2$ и $b$ так, чтобы $L=0$ для любой точки в нашем наборе данных.

Чтобы применить градиентный спуск, нам нужны производные $L$ по $w_1$, $w_2$ и $b$.

$$
\frac{\partial L}{\partial w_k} =
\frac{2}{N} \sum_{i=1}^N \left[\sigma(x_{i1} w_1 + x_{i2} w_2 + b) - y_i\right] \sigma'(x_{i1} w_1 + x_{i2} w_2 + b) x_{ik}
$$

$$
\frac{\partial L}{\partial b} =
\frac{2}{N} \sum_{i=1}^N \left[\sigma(x_{i1} w_1 + x_{i2} w_2 + b) - y_i\right] \sigma'(x_{i1} w_1 + x_{i2} w_2 + b)
$$

Здесь $\sigma'()$ — производная сигмоидной функции:

$$
\sigma'(x) = \left(\frac{1}{1+e^{-x}}\right)'=(-1)\frac{1}{(1+e^{-x})^2}(-1) e^{-x}=
\frac{e^{-x}}{1+e^{-x}}\frac{1}{1+e^{-x}}=
\frac{1}{1+e^x}\frac{1}{1+e^{-x}}=\sigma(x)\sigma(-x)
$$

Наконец, получаем:

$$
\frac{\partial L}{\partial w_k} =
\frac{2}{N} \sum_{i=1}^N \left[\sigma(y_\text{logits}) - y_i\right] \sigma(y_\text{logits})\sigma(-y_\text{logits}) x_{ik}
$$

$$
\frac{\partial L}{\partial b} =
\frac{2}{N} \sum_{i=1}^N \left[\sigma(y_\text{logits}) - y_i\right] \sigma(y_\text{logits})\sigma(-y_\text{logits})
$$
где $y_\text{logits}=x_{i1} w_1 + x_{i2} w_2 + b$.

Эти $\partial L/\partial w_k$ ($k=1,2$) и $\partial L/\partial b$ являются компонентами градиента $\nabla L$.

Теперь мы можем использовать этот градиент для организации обновлений параметров нейрона.

Мы создадим новую версию класса Neuron.


In [ ]:
class Neuron:
    """
    Neuron with sigmoid activation and gradient with respect of MSE
    """
    def __init__(self, w, b):
        self.w = np.array(w)
        self.b = b

    def __str__(self):
        """String reprsentation to print weights"""
        return f"w={self.w}, b={self.b}"

    def sigmoid(x):
        """Activation function used for training"""
        return 1.0 / (1.0 + np.exp(-x))

    def stepfunc(y):
        """Activation of the output layer for prediction mode"""
        return (y >= 0).astype(int)

    def __call__(self, x, logits=False):
        """Feed-forward of inputs"""
        y0 = np.array(x) @ self.w + self.b
        if logits:
            return y0
        else:
            return Neuron.sigmoid(y0)

    def predict(self, x):
        """Do prediction"""
        y0 = self.__call__(x, logits=True)
        return Neuron.stepfunc(y0)

    def mae(self, x, y_true):
        """Mean absolute error, will be used as a metric reporting performance of the neuron"""
        y_pred = self.predict(x)
        return np.mean(np.abs((y_pred - y_true)))

    def grad(self, x, y_true):
        """Gradinet with respect to MSE loss function"""
        y0 = self.__call__(x, logits=True)
        sig_plus = Neuron.sigmoid(y0)
        sig_minus = Neuron.sigmoid(-y0)
        err = sig_plus - y_true
        tmp = err * sig_plus * sig_minus

        gr = []
        for xj in x.T:
            gr.append(2 * np.mean(tmp * xj))
        gr.append(2 * np.mean(tmp))

        return np.array(gr)

Используя этот класс, создадим нейрон и обучим его находить верхнюю правую группу точек в нашем наборе данных.


In [ ]:
import numpy as np
rng = np.random.default_rng(seed=0)

# True class labels can be easily derived manually
y_true = [1 if x[0] > 0 and x[1] > 5 else 0 for x in data]

# The neuron with random initial values of parameters
nrn = Neuron(w = rng.uniform(size=(2,)), b = rng.uniform())
print(nrn)

Это процедура обучения. Мы вычисляем градиенты и применяем шаг градиентного спуска ко всему набору данных сразу, поскольку набор данных невелик.

Каждый шаг называется эпохой. Мы повторяем несколько эпох, пока средняя абсолютная ошибка (MAE) не станет нулевой.

In [ ]:
epochs =  500
alpha = 1

train_curve = []
for epoch in range(epochs):
    grad = nrn.grad(data, y_true)
    nrn.w -= alpha * grad[:-1]
    nrn.b -= alpha * grad[-1]
    train_curve.append(nrn.mae(data, y_true))

train_curve[-10:]
print(nrn)

Этот график называется кривой обучения (learning curve). Он показывает процесс обучения сети.


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots()

ax.plot(np.arange(epochs), train_curve)
ax.set_xlabel("epoch")
ax.set_ylabel("mae");

Теперь наша сеть полностью обучена, и мы можем проверить, как она работает.

Это предсказания для набора данных.

Поскольку мы рассматриваем игрушечный пример, мы не разделяем набор данных на части для обучения, валидации и тестирования.

In [ ]:
y_pred = nrn.predict(data)
print(y_pred)

Для лучшей иллюстрации мы вычислим плотную сетку на плоскости и вычислим класс для каждой точки сетки.

In [ ]:
x_mesh = np.linspace(-1.7, 1.7, 100)  # x coordinates
y_mesh = np.linspace(3.3, 6.7, 100)    # y coordinates
X, Y = np.meshgrid(x_mesh, y_mesh)    # combine each one with each one
data_mesh = np.array([X.reshape(-1), Y.reshape(-1)]).T  # create a dataset that covers the whole plane
y_mesh = nrn.predict(data_mesh)

In [ ]:
import matplotlib.pyplot as plt

colors = ['r', 'b']

fig, ax = plt.subplots(figsize=(5,5))
ax.scatter(data_mesh[:, 0], data_mesh[:, 1], color=[colors[y] for y in y_mesh], s=0.1, alpha=0.6)
ax.scatter(data[:, 0], data[:, 1], color=[colors[y] for y in y_pred])
ax.axline((-0.5, 6.5), (1.5, 4.5), color='k')
ax.set_xlabel(r"$x_1$")
ax.set_ylabel(r"$x_2$")
ax.grid()

Светло‑красные и светло‑синие области показывают, где сеть нашла границу между двумя классами.

Чёрная линия — это граница, которую мы определили, просто наблюдая распределение точек.

Границы различаются, но обе являются подходящими.


## Тензоры

Данные, обрабатываемые сетями, собираются как многомерные массивы.

Ранее мы рассматривали только двумерные: каждый столбец — это некоторый признак, а каждая строка — определенная сущность.

Но что, если мы хотим обрабатывать изображения?

Каждое изображение само по себе является массивом. Если изображение в оттенках серого, это двумерный массив.

Каждый элемент представляет пиксель.

Для цветных изображений одного элемента для пикселя недостаточно. Цвета обычно кодируются тремя числами, представляющими красный, зеленый и синий цвета.

Таким образом, цветные изображения представляют собой трехмерные массивы: каждый сайт с индексами $i$ и $j$ содержит еще один массив цветовых каналов.

Если мы хотим обрабатывать изображения сетью, мы собираем их в пакет (batch). Таким образом, пакет цветных изображений является четырехмерным массивом.

В науке о данных многомерные массивы часто называют тензорами.

Тензоры «прямоугольны»: вдоль каждой оси каждый элемент имеет одинаковый размер.

Обратите внимание, что математический «тензор» — это более сложный объект, чем просто массив чисел. Кроме того, эти числа должны обладать определенными свойствами.

К счастью, математические тензоры нам не понадобятся.

Мы будем работать с нейронными сетями, используя фреймворк Tensorflow https://www.tensorflow.org

Все данные, используемые для создания сети, будут преобразованы в тензоры TF.

Эти тензоры совместимы с массивами numpy, поэтому мы можем легко преобразовывать тензоры в массивы numpy и обратно.

Количество измерений тензора называется рангом тензора. Также ранг называется порядком, степенью или n-мерностью.

Скаляры рассматриваются как тензоры ранга 0.


In [ ]:
import tensorflow as tf

rank0_tensor = tf.constant(13.0)
print(rank0_tensor)

Ранг тензора можно проверить с помощью функции `tf.rank`.

Она также возвращает тензор, содержимое которого задаёт требуемый ранг.


In [ ]:
print(tf.rank(rank0_tensor))

Чтобы получить более удобный для чтения вид, можно извлечь числовое значение с помощью метода `.numpy()`:


In [ ]:
print(tf.rank(rank0_tensor).numpy())

Вектор — это тензор ранга 1.


In [ ]:
rank1_tensor = tf.constant([12.0, 13.0, 14.0, 15.0])
print(rank1_tensor)
print(tf.rank(rank1_tensor).numpy())

Матрица — это тензор ранга 2:


In [ ]:
rank2_tensor = tf.constant([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])
print(rank2_tensor)
print(tf.rank(rank2_tensor).numpy())

При создании тензора тип данных можно указать с помощью параметра `dtype`:


In [ ]:
x = tf.constant(3, dtype=tf.int32)
y = tf.constant(3, dtype=tf.float32)
print(x)
print(y)

Число 32 в обозначении типа означает количество бит, необходимых для хранения числа.

TensorFlow поддерживает много различных типов данных, которые в основном соответствуют типам данных NumPy.


Тензоры поддерживают базовые математические операции, включая сложение, поэлементное умножение и умножение матриц.


In [ ]:
a = tf.constant([[1, 2], [3, 4]])
b = tf.constant([[1, 1], [1, 1]])

print(tf.add(a, b), "\n")
print(tf.multiply(a, b), "\n")
print(tf.matmul(a, b), "\n")

In [ ]:
print(a + b, "\n") # element-wise addition
print(a * b, "\n") # element-wise multiplication
print(a @ b, "\n") # matrix multiplication

Как и в случае массивов NumPy, к элементам тензора можно обращаться с помощью индексации:


In [ ]:
print(a[0, 0])
print(a[0, 1])
print(a[1, 0])
print(a[1, 1])

## Двухслойная сеть прямого распространения. Обучение с TensorFlow

Выше мы создали сеть, которая может различать классы, лежащие вдоль диагоналей: верхний правый и нижний левый углы — класс 1, а два других — класс 0.

Эта сеть была настроена вручную.

Теперь мы собираемся создать сеть с помощью tensorflow и обучить ее распознавать эти классы.

Tensorflow предлагает множество способов создания сети: можно использовать «крупные» строительные блоки для быстрого и легкого построения, а можно углубиться в детали и использовать простейшие элементы для тонкой настройки.

Мы будем использовать первую стратегию.

Ниже мы импортируем tensorflow и создаем последовательную модель (sequential). Нужно просто перечислить все необходимые слои один за другим. Затем tensorflow соединит слои соответствующим образом.

Такой способ построения сети заимствован из библиотеки Keras.

Раньше это была отдельная библиотека для моделирования нейронных сетей, а теперь из-за своего удобства она стала частью tensorflow.

Наша сеть будет иметь точно такую же структуру, как и выше.

У нее будет один скрытый слой из двух нейронов и выходной слой с одним нейроном.

Слои, где каждый вход соединен с каждым нейроном, в tensorflow называются плотными слоями (Dense).

In [ ]:
import tensorflow as tf
import numpy as np

kernel_initializer = tf.keras.initializers.RandomUniform(minval=0, maxval=1, seed=1)
activation = 'sigmoid'

model = tf.keras.models.Sequential([
  tf.keras.layers.Dense(2, activation=activation, kernel_initializer = kernel_initializer),
  tf.keras.layers.Dense(1, activation=activation, kernel_initializer = kernel_initializer)
])

Первый слой (скрытый) содержит 2 нейрона, а второй — 1 нейрон.

Мы указываем, какую активацию мы хотим и как будут инициализироваться веса нейронов.

Мы берем сигмоиду в качестве функции активации и используем генератор случайных равномерных чисел для инициализации весов.

Инициализатор смещений слоев не указан. В этом случае они будут инициализированы нулями.

Теперь нам нужно подготовить сеть, инкапсулированную внутри объекта `model`, к обучению.

Это делается с помощью метода `.compile`:

In [ ]:
loss = tf.keras.losses.MeanSquaredError()
optim = tf.keras.optimizers.SGD(learning_rate=1.0) # Stochastic Gradient Descent

model.compile(optimizer=optim, loss=loss, metrics=['accuracy'])

Мы берем среднеквадратичную ошибку в качестве функции потерь, которая будет минимизироваться в процессе обучения, а метод оптимизации — стохастический градиентный спуск.

Напомним: стохастический здесь означает, что весь набор обучающих данных разбивается на маленькие мини-пакеты (mini-batches), и шаги градиентного спуска выполняются для этих мини-пакетов один за другим.

Когда все мини-пакеты показаны сети, эпоха заканчивается.

Затем начинается новая эпоха, когда снова мини-пакеты показываются один за другим и параметры обновляются.

Это делается потому, что обычно набор данных может быть огромным и не помещаться в память компьютера целиком.

Также мы указываем так называемую метрику. Метрика не используется для вычисления градиентов и обновления параметров. Она вычисляется в конце каждой эпохи для оценки достигнутой производительности.

Нам также нужны желаемые метки классов. Для нашего простого набора данных метки могут быть сгенерированы вручную.

Мы помечаем точки в верхнем правом и нижнем левом углах как класс 1, а остальные — как класс 0.


In [ ]:
y_true = np.array([1 if (x[0] > 0 and x[1] > 5) or (x[0] < 0 and x[1] < 5) else 0 for x in data])

Перед началом обучения разделим данные на обучающую и тестовую части.


In [ ]:
print(data.shape)

In [ ]:
n_test = 100
X_train = data[:-n_test]
y_train = y_true[:-n_test]

X_test = data[-n_test:]
y_test = y_true[-n_test:]

Выделять валидационную часть явно не требуется.

Вместо этого мы укажем обучающей процедуре сделать это автоматически.

Обучение выполняется вызовом метода `.fit`.


In [ ]:
epochs = 100 # nunber of epochs to do training
validation_split = 0.2  # 20% of data will be used for validation only
hist = model.fit(X_train, y_train, epochs=epochs, verbose=1, validation_split=validation_split);

Метод `.fit` возвращает объект history, который содержит список вычисленных значений функции потерь и метрик.

Их зависимость от номера эпохи называется кривыми обучения.


In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(10, 3))

ax = axs[0]
ax.plot(range(epochs), hist.history['loss'], label='loss')
ax.plot(range(epochs), hist.history['val_loss'], label='val_loss')
ax.set_ylabel('loss')

ax = axs[1]
ax.plot(range(epochs), hist.history['accuracy'], label='accuracy');
ax.plot(range(epochs), hist.history['val_accuracy'], label='val_accuracy');
ax.set_ylabel('accuracy')

for ax in axs:
    ax.grid()
    ax.set_xlabel('epoch')
    ax.legend()

Наша сеть способна достичь точности 100%.

Проверим это на тестовых данных.

Вот как можно использовать сеть для предсказания:


In [ ]:
y_pred = model(X_test, training=False)
print(y_pred[:10])

Значения, которые мы наблюдаем, являются выходами сигмоиды. Нам нужно преобразовать их в 0 и 1: если $y \geq 0.5$, записываем 1, иначе — 0.

Также обратите внимание на параметр `training`.

Для нашей сети это несущественно. Но существуют архитектуры, которые ведут себя по-разному в режиме обучения и в режиме предсказания. Поэтому лучше всегда явно задавать `training=False`, когда мы используем сеть для предсказания.


In [ ]:
y_pred = [1 if y[0] >= 0.5 else 0 for y in model(X_test, training=False)]

Снова построим график нашего набора данных: левая панель будет окрашена в соответствии с желаемой разметкой классов, а правая — в соответствии с предсказанием сети.

Кроме того, мы покроем плоскость плотной сеткой точек и вычислим предсказанные классы для каждой точки.

Если изобразить эти точки соответствующими цветами, получится окрашенная область, показывающая, как сеть разделяет классы.


In [ ]:
x_mesh = np.linspace(-1.7, 1.7, 100)  # x coordinates
y_mesh = np.linspace(3.3, 6.7, 100)    # y coordinates
X, Y = np.meshgrid(x_mesh, y_mesh)    # combine each one with each one
data_mesh = np.array([X.reshape(-1), Y.reshape(-1)]).T  # create a dataset that covers the whole plane
y_mesh = [1 if y[0] >= 0.5 else 0 for y in model(data_mesh, training=False)]

In [ ]:
import matplotlib.pyplot as plt

colors = ['r', 'b']

fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(11,5))
ax = axs[0]
ax.scatter(X_test[:, 0], X_test[:, 1], color=[colors[y] for y in y_test])
ax.set_title("Colors according to the manual class labeling")

ax = axs[1]
ax.scatter(data_mesh[:, 0], data_mesh[:, 1], color=[colors[y] for y in y_mesh], s=0.1, alpha=0.6)
ax.scatter(X_test[:, 0], X_test[:, 1], color=[colors[y] for y in y_pred])
ax.set_title("Network prediction")

for ax in axs:
    ax.set_xlabel(r"$x_1$")
    ax.set_ylabel(r"$x_2$")
    ax.grid()
    ax.set_xlim([-1.7, 1.7])
    ax.set_ylim([3.3, 6.7])

## Широкая и глубокая сеть: сумматор целых чисел

Теперь мы создадим две сети, которые обучаются суммировать целые числа.

Для сравнения рассмотрим двухслойную сеть с широким скрытым слоем и другую глубокую сеть, включающую четыре не очень больших слоя.

Будет показано, что глубокая сеть имеет гораздо меньше обучаемых параметров и работает лучше, чем широкая.

Размерность входных данных у нас равна двум - два числа подаются на вход. Непосредственная обработка этих чисел получается плохо, поэтому мы искусственно повысим размерность: преобразуем числа двоичную форму и ответ также будем получать в двоичном виде.

Функция `int2bin` преобразует десятичные числа в двоичную строку фиксированной длины.

Она использует встроенную функцию `bin`, но расширяет ее вывод до фиксированного размера.

In [ ]:
bin(0), bin(1), bin(2), bin(7), bin(9), bin(17)

In [ ]:
import numpy as np
rng = np.random.default_rng()

def int2bin(d, n_bits):
    """Convert integer to binary number having fixed number of difits n_bits"""
    b = bin(d)[2:]  # remove prefix '0b'
    add_zeros = n_bits - len(b)
    if add_zeros > 0:
        b = add_zeros * '0' + b
    return b

print(int2bin(7, 8))
print(int2bin(9, 8))
print(int2bin(17, 8))

Функция `int2list` принимает строку, созданную функцией `int2bin`, и преобразует её в список двоичных цифр.


In [ ]:
def int2list(d, n_bits):
    """Converts integer to list of binary numbers that correspond to its binary representation"""
    s = int2bin(d, n_bits)
    return [int(x) for x in s]

print(int2list(7, 8))
print(int2list(9, 8))
print(int2list(17, 8))

Функция `get_values` генерирует случайный список целых чисел.


In [ ]:
def get_values(N, n_bits, rng):
    """Generate random list of numbers"""
    nums = list(range(2**n_bits))
    rng.shuffle(nums)
    return nums[:N]

print(get_values(10, 6, rng))

Теперь мы готовы создать набор пар целых чисел и их сумм. `N` — это число целых чисел, а `n_bits` — максимальная длина рассматриваемых чисел.


In [ ]:
N = 128
n_bits = 8

values1 = get_values(N, n_bits, rng)
values2 = get_values(N, n_bits, rng)
print(values1[:10])
print(values2[:10])

data_orig = []
for i in range(N):
    for j in range(i, N):
        a = values1[i]
        b = values2[j]
        data_orig.append([a, b, a + b])
rng.shuffle(data_orig)
data_orig = np.array(data_orig)

print(data_orig.shape)
print(data_orig[:10])

Целые числа и их суммы теперь нужно преобразовать в двоичную форму. Списки, представляющие слагаемые, объединяются.

In [ ]:
X_data = []
Y_data = []
for abc in data_orig:
    a, b, c = abc
    X_data.append(int2list(a, n_bits) + int2list(b, n_bits))
    Y_data.append(int2list(c, n_bits + 1))

X_data = np.array(X_data)
Y_data = np.array(Y_data)

print(X_data.shape)
print(X_data[:10], "\n")
print(Y_data.shape)
print(Y_data[:10])

Разделим набор данных на обучающую и тестовую части.


In [ ]:
p_test = 0.1
n_test = int(p_test * len(X_data))
X_train = X_data[:-n_test]
Y_train = Y_data[:-n_test]

X_test = X_data[-n_test:]
Y_test = Y_data[-n_test:]

print(X_train.shape)
print(X_test.shape)

Гиперпараметры сети задаются следующим образом.


In [ ]:
import tensorflow as tf

epochs = 300 # nunber of epochs to train
validation_split = 0.2  # 20% of data will be used for validation only
learning_rate = 1e-3

activation = tf.keras.activations.hard_sigmoid

Обратите внимание, что мы собираемся использовать упрощённую сигмоиду, так называемую hard sigmoid. Это нужно потому, что мы хотим получать на выходе точные 0 и 1, а обычная сигмоида лишь асимптотически приближается к 0 и 1.


In [ ]:
import matplotlib.pyplot as plt

xs = tf.constant(np.linspace(-6, 6, 100), dtype = tf.float32)
ys1 = tf.keras.activations.hard_sigmoid(xs)
ys2 = tf.keras.activations.sigmoid(xs)

fig, ax = plt.subplots()

ax.plot(xs.numpy(), ys1.numpy(), label='hard sigmoid')
ax.plot(xs.numpy(), ys2.numpy(), label='sigmoid')
ax.legend()
ax.grid()

Сначала наша модель будет широкой. Её скрытый слой большой, а выходной слой содержит `n_bits+1` нейронов, поскольку именно таков размер требуемых выходных массивов.


In [ ]:
model_wide = tf.keras.models.Sequential([
  tf.keras.layers.Dense(10*n_bits, activation=activation),
  tf.keras.layers.Dense(n_bits+1, activation=activation)
])

loss = tf.keras.losses.MeanSquaredError()
optim = tf.keras.optimizers.Adam(learning_rate=learning_rate)

model_wide.compile(optimizer=optim, loss=loss, metrics=['accuracy'])

model_wide.evaluate(X_train, Y_train, verbose=0)  # need this to create inner structure
model_wide.summary()

Мы применили выше метод `.evaluate`, потому что внутренняя структура модели фактически создаётся только при её первом вызове. После этого мы выводим сводку модели.

Мы видим, что широкая модель имеет `2089` обучаемых параметров. Общая идея состоит в том, что чем меньше параметров, тем быстрее обычно проходит обучение (разумеется, если при меньшем числе параметров качество остаётся хорошим).

Обучим модель и посмотрим, как она работает.


In [ ]:
hist_wide = model_wide.fit(X_train, Y_train, epochs=epochs,
                           verbose=1, validation_split=validation_split)

Кривые обучения имеют следующий вид.


In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(10, 3))

ax = axs[0]
ax.plot(range(epochs), hist_wide.history['loss'], label='loss')
ax.plot(range(epochs), hist_wide.history['val_loss'], label='val_loss')
ax.set_ylabel('loss')
ax.set_yscale('log')

ax = axs[1]
ax.plot(range(epochs), hist_wide.history['accuracy'], label='accuracy');
ax.plot(range(epochs), hist_wide.history['val_accuracy'], label='val_accuracy');
ax.set_ylabel('accuracy')

for ax in axs:
    ax.grid()
    ax.set_xlabel('epoch')
    ax.legend()

Наконец, протестируем модель.


In [ ]:
model_wide.evaluate(X_test, Y_test)

Теперь рассмотрим глубокую сеть. Она содержит четыре не слишком широких слоя.


In [ ]:
model_deep = tf.keras.models.Sequential([
  tf.keras.layers.Dense(2*n_bits, activation=activation),
  tf.keras.layers.Dense(n_bits+1, activation=activation),
  tf.keras.layers.Dense(n_bits+1, activation=activation),
  tf.keras.layers.Dense(n_bits+1, activation=activation)
])

loss = tf.keras.losses.MeanSquaredError()
optim = tf.keras.optimizers.Adam(learning_rate=learning_rate)

model_deep.compile(optimizer=optim, loss=loss, metrics=['accuracy'])

model_deep.evaluate(X_train, Y_train, verbose=0)  # need this to create inner structure
model_deep.summary()

Обратите внимание, что у глубокой модели более чем в три раза меньше параметров. Посмотрим, как она работает.


In [ ]:
hist_deep = model_deep.fit(X_train, Y_train, epochs=epochs,
                           verbose=1, validation_split=validation_split)

На рисунках ниже сравниваются кривые обучения для широкой и глубокой моделей. Мы видим, что глубокая модель работает лучше. Её точность заметно выше.


In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(nrows=2, ncols=2, figsize=(10, 6), sharex=True)

ax = axs[0,0]
ax.plot(range(epochs), hist_wide.history['loss'], label='loss')
ax.plot(range(epochs), hist_wide.history['val_loss'], label='val_loss')
ax.set_ylabel('loss')
ax.set_yscale('log')
ax.set_title('wide')

ax = axs[0,1]
ax.plot(range(epochs), hist_wide.history['accuracy'], label='accuracy');
ax.plot(range(epochs), hist_wide.history['val_accuracy'], label='val_accuracy');
ax.set_ylabel('accuracy')
ax.set_title('wide')

ax = axs[1,0]
ax.plot(range(epochs), hist_deep.history['loss'], label='loss')
ax.plot(range(epochs), hist_deep.history['val_loss'], label='val_loss')
ax.set_ylabel('loss')
ax.set_yscale('log')
ax.set_title('deep')

ax = axs[1,1]
ax.plot(range(epochs), hist_deep.history['accuracy'], label='accuracy');
ax.plot(range(epochs), hist_deep.history['val_accuracy'], label='val_accuracy');
ax.set_ylabel('accuracy')
ax.set_ylim([0.8, 1])
ax.set_title('deep')

for ax in axs.ravel():
    ax.grid()
    ax.legend()

axs[1,0].set_xlabel('epoch')
axs[1,1].set_xlabel('epoch');

In [ ]:
model_wide.evaluate(X_test, Y_test)

In [ ]:
model_deep.evaluate(X_test, Y_test)

Мы видим явное отличие. Глубокая сеть имеет меньше параметров и при этом работает лучше.
